# Weak-label exploration v1 — urgency on customer tickets

Notebook-first exploratory slice. The goal is a **first, defensible** weak label for ticket urgency on `inbound = True` customer tweets from `data/raw/twcs.csv`.

Scope of this notebook:
1. Load raw CSV
2. Select the 7 relevant columns
3. Base text cleaning
4. Filter to `inbound = True`
5. Define a simple urgent / normal weak-label rule
6. Apply it, inspect class distribution + examples
7. Save `data/processed/customer_tickets_weak_labeled_v1.csv`

Out of scope: engineered features, training, embeddings, pipeline integration.

## 1. Imports and paths

In [1]:
import re
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_CSV       = PROJECT_ROOT / "data" / "raw" / "twcs.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Optional cap for quick iteration. Set to None for the full dataset.
NROWS = None

RELEVANT_COLUMNS = [
    "tweet_id",
    "author_id",
    "inbound",
    "created_at",
    "text",
    "response_tweet_id",
    "in_response_to_tweet_id",
]
RAW_CSV, PROCESSED_DIR, NROWS

(WindowsPath('C:/Users/Jamal/Documents/bootcamps/AIE/Week 3/Decision-Intelligence-Assistant/data/raw/twcs.csv'),
 WindowsPath('C:/Users/Jamal/Documents/bootcamps/AIE/Week 3/Decision-Intelligence-Assistant/data/processed'),
 None)

## 2. Load + select columns + base cleaning

Base cleaning mirrors the pipeline: drop null text, cast to string, strip, drop empties, coerce `inbound` to real bool.

In [2]:
df = pd.read_csv(RAW_CSV, nrows=NROWS)
print("raw rows:", len(df))
df = df[RELEVANT_COLUMNS].copy()

df = df.dropna(subset=["text"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""].copy()

if df["inbound"].dtype != bool:
    df["inbound"] = df["inbound"].map({True: True, False: False, "True": True, "False": False})
    df = df.dropna(subset=["inbound"]).copy()
    df["inbound"] = df["inbound"].astype(bool)

print("rows after base cleaning:", len(df))
df.head(2)

raw rows: 2811774


rows after base cleaning: 2811774


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0


## 3. Filter to customer tickets (`inbound = True`)

In [3]:
customer = df[df["inbound"]].copy()
print("customer tickets:", len(customer))
customer.head(2)

customer tickets: 1537843


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0


## 4. Weak-label rule v1

**Why this rule.** We want a *first* signal that is cheap, transparent, and defensible in review. A keyword + punctuation heuristic is the simplest thing that could work:

- **Keyword groups** target the language customers use when something is blocking or financial: `refund / charged / billing / payment / overcharged`, broken-service wording (`broken / not working / down / error / issue / problem`), access loss (`locked / can't log in / access`), explicit urgency (`urgent / asap / immediately / help`), and escalation signals (`cancel / missing / delayed`).
- **Punctuation intensity** — repeated `!!` or `???` — captures affect that isn't keyword-based.
- Word-boundary anchors (`\b`) reduce false positives like `downtown` matching `down`.
- Case-insensitive.

Known limits: this is a **binary proxy**, not ground truth. It will over-call some support tweets (false positives on `help` in casual use) and miss subtle urgency. A later slice will compare against a held-out human-labeled sample.

In [4]:
URGENT_KEYWORD_PATTERNS = [
    r"\brefund(ed|s)?\b",
    r"\bcharged?\b",
    r"\bbilling\b",
    r"\bpayment(s)?\b",
    r"\bovercharged?\b",
    r"\bbroken\b",
    r"\bnot working\b",
    r"\bdown\b",
    r"\berror(s)?\b",
    r"\bissue(s)?\b",
    r"\bproblem(s)?\b",
    r"\block(ed)?\b",
    r"\bcan'?t log ?in\b",
    r"\bcannot log ?in\b",
    r"\baccess\b",
    r"\burgent\b",
    r"\basap\b",
    r"\bimmediately\b",
    r"\bhelp\b",
    r"\bcancel(lation|led|s)?\b",
    r"\bmissing\b",
    r"\bdelayed\b",
]
URGENT_REGEX = re.compile("|".join(URGENT_KEYWORD_PATTERNS), re.IGNORECASE)
PUNCT_REGEX  = re.compile(r"!{2,}|\?{2,}")

def weak_label_urgency(text: str) -> str:
    if URGENT_REGEX.search(text) or PUNCT_REGEX.search(text):
        return "urgent"
    return "normal"

# Sanity check
for t in [
    "my account is locked and I can't log in",
    "please refund my payment ASAP",
    "great service today",
    "help!!! my flight is delayed",
]:
    print(f"{weak_label_urgency(t):>6}  |  {t}")

urgent  |  my account is locked and I can't log in
urgent  |  please refund my payment ASAP
normal  |  great service today
urgent  |  help!!! my flight is delayed


## 5. Apply label

In [5]:
customer["priority_label"] = customer["text"].map(weak_label_urgency)
customer["priority_label"].value_counts(dropna=False)

priority_label
normal    1167203
urgent     370640
Name: count, dtype: int64

## 6. Distribution + example inspection

In [6]:
dist = customer["priority_label"].value_counts(normalize=True).round(4)
print("class distribution (proportion):")
print(dist)
print()
print("absolute counts:")
print(customer["priority_label"].value_counts())

class distribution (proportion):
priority_label
normal    0.759
urgent    0.241
Name: proportion, dtype: float64

absolute counts:
priority_label
normal    1167203
urgent     370640
Name: count, dtype: int64


In [7]:
print("--- urgent examples ---")
for t in customer.loc[customer["priority_label"] == "urgent", "text"].head(5).tolist():
    print("-", t[:180])
print()
print("--- normal examples ---")
for t in customer.loc[customer["priority_label"] == "normal", "text"].head(5).tolist():
    print("-", t[:180])

--- urgent examples ---
- actually that's a broken link you sent me and incorrect information https://t.co/V4yfrHR8VI
- somebody from @VerizonSupport please help meeeeee 😩😩😩😩 I'm having the worst luck with your customer service
- @VerizonSupport What else can I provide? They refuse to help me because they cannot validate the account...
- .@VerizonSupport @115725 @115726                                                 &gt;All of VERIZON IS DOWN&lt;
When can we expect a fix ?
- @ChipotleTweets Thank you @ChipotleTweets for resolving my issue so quickly!! Y’all are the best ☺️ #fanforlife

--- normal examples ---


-

 @sprintcare and how do you propose we do that
- @sprintcare I have sent several private messages and no one is responding as usual
- @sprintcare I did.
- @sprintcare is the worst customer service
- @sprintcare You gonna magically change your connectivity for me and my whole family ? 🤥 💯


## 7. Save labeled CSV to `data/processed/`

In [8]:
out_path = PROCESSED_DIR / "customer_tickets_weak_labeled_v1.csv"
customer.to_csv(out_path, index=False)
print("wrote:", out_path)
print("rows:", len(customer))
print("columns:", list(customer.columns))

wrote: C:\Users\Jamal\Documents\bootcamps\AIE\Week 3\Decision-Intelligence-Assistant\data\processed\customer_tickets_weak_labeled_v1.csv
rows: 1537843
columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'priority_label']
